# DSS Desa Wisata Indonesia (1.336 Desa) — TF-IDF, LR, Calibrated SVM, Random Forest & Permutation SHAP

**Rancangan untuk revisi manuskrip IJITCS** · Data: **987 di luar benchmark, 315 Maju, 34 Mandiri**.

Notebook ini mempertahankan seluruh komponen DSS: pembersihan dan rekayasa fitur profil, TF-IDF, perbandingan **Logistic Regression**, **Calibrated Linear SVM**, **Random Forest**, validasi silang, optimasi threshold **hanya dari data training**, evaluasi *holdout*, penilaian kesiapan menuju Mandiri, **Permutation SHAP pada fungsi skor DSS sesungguhnya** (tanpa model XGBoost tambahan), dan rekomendasi kesenjangan benchmark.

**Arsitektur dua tahap (hierarchical classification):**

1. **Tahap A:** klasifikasi *tercakup dalam benchmark tingkat lanjut* (`Maju/Mandiri`, label 1 atau 2) versus **di luar daftar benchmark** (label 0, **bukan** berarti resmi Rintisan/Berkembang).
2. **Tahap B:** di antara desa berlabel Maju/Mandiri, klasifikasi **Mandiri vs Maju**.
3. `skor_kemiripan_mandiri = 100 × P(benchmark tingkat lanjut | X) × P(Mandiri | benchmark tingkat lanjut, X)`.
4. Dua probabilitas berasal dari **model yang benar-benar dipilih dan digunakan**; SHAP menjelaskan fungsi perkalian yang sama.

> **Batasan label:** pada ketiga CSV, `status_terverifikasi_JADESTA = BELUM`. Karena itu, keluaran adalah **skor kemiripan terhadap label benchmark yang diberikan**, **bukan penetapan status resmi atau probabilitas empiris menjadi desa mandiri**. Tinjau/verifikasi daftar benchmark sebelum klaim definitif.

> **Keamanan evaluasi:** model dan threshold dipilih dengan *cross-validation / out-of-fold* pada data latih; set uji tetap terkunci. Hasil penilaian seluruh desa menggunakan model yang **dilatih ulang pada seluruh data** dan tidak boleh disajikan sebagai evaluasi luar-sampel.

## Tahap 0 — Persiapan Colab dan konfigurasi

Unggah tiga CSV bernama persis `DESA_DI_LUAR_BENCHMARK_987.csv`, `DESA_MAJU_315.csv`, dan `DESA_MANDIRI_34.csv` melalui dialog unggah Colab. Bila berjalan di Jupyter lokal dan file sudah ada, pengunggahan dilewati.

Untuk menguji secara cepat, ubah `FAST_MODE = True` (atau set variabel lingkungan `DSS_FAST_MODE=1`). **Gunakan `False` untuk hasil penelitian**.

In [ ]:
import os, sys, re, json, zipfile, warnings, random, platform
from pathlib import Path
from urllib.parse import urlparse
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from sklearn import __version__ as sklearn_version
from sklearn.base import clone
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, cross_val_predict
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, average_precision_score, confusion_matrix,
                             ConfusionMatrixDisplay, classification_report, brier_score_loss)
from scipy.stats import beta

SEED = 42
np.random.seed(SEED)
random.seed(SEED)
FAST_MODE = os.environ.get('DSS_FAST_MODE', '0') == '1'  # Default: MODE PENELITIAN
CV_FOLDS = 3 if FAST_MODE else 5
RF_TREES = 55 if FAST_MODE else 160
SHAP_BG_PER_CLASS = (3, 3, 3) if FAST_MODE else (10, 6, 4)
SHAP_EXPLAIN_PER_CLASS = (3, 2, 1) if FAST_MODE else (12, 9, 6)
SHAP_N_PERMUTATIONS = 1 if FAST_MODE else 2

OUT_DIR = Path('/content/hasil_dss_desa_wisata' if Path('/content').exists() else 'hasil_dss_desa_wisata')
OUT_DIR.mkdir(parents=True, exist_ok=True)
FILES = {
    0: 'DESA_DI_LUAR_BENCHMARK_987.csv',
    1: 'DESA_MAJU_315.csv',
    2: 'DESA_MANDIRI_34.csv',
}
LABEL_NAMES = {0: 'Di Luar Benchmark', 1: 'Maju', 2: 'Mandiri'}
print('FAST_MODE:', FAST_MODE, '| CV_FOLDS:', CV_FOLDS, '| output:', OUT_DIR.resolve())
print('scikit-learn:', sklearn_version)

## Tahap 1 — Upload, gabungkan tiga data, dan audit label

Tidak menggunakan file label eksternal: kategori ditetapkan **berdasarkan file sumber**. Label dan informasi sumber tidak digunakan sebagai fitur prediktif. Validasi dilakukan terhadap *matching_key* sebagai identitas unik.

In [ ]:
missing_files = [name for name in FILES.values() if not Path(name).exists()]
if missing_files:
    try:
        from google.colab import files as colab_files
        print('Silakan unggah semua CSV berikut:', missing_files)
        colab_files.upload()
    except ImportError as exc:
        raise FileNotFoundError('Salin file berikut ke folder kerja: ' + ', '.join(missing_files)) from exc

frames = []
for label, filename in FILES.items():
    if not Path(filename).exists():
        raise FileNotFoundError('CSV tidak tersedia: ' + filename)
    part = pd.read_csv(filename, dtype=str, keep_default_na=False, encoding='utf-8-sig')
    part['label_3kelas'] = label
    part['status_benchmark'] = LABEL_NAMES[label]
    part['file_sumber_label'] = filename
    frames.append(part)
    print(filename, '->', len(part), 'baris')

df_raw = pd.concat(frames, ignore_index=True)
assert df_raw.shape[0] == 1336, f'Jumlah data berubah: {len(df_raw)}. Audit sumber sebelum melanjutkan.'
assert df_raw['label_3kelas'].value_counts().to_dict() == {0: 987, 1: 315, 2: 34}
required_columns = ['Desa Wisata','Kabupaten','Deskripsi','Fasilitas','Atraksi','homestay',
                    'pakert wisata','sovenir','HP','IG','FB','Email','Website','Twitter','matching_key']
missing_cols = sorted(set(required_columns) - set(df_raw.columns))
assert not missing_cols, f'Kolom hilang: {missing_cols}'
assert (df_raw['matching_key'].astype(str).str.strip() != '').all(), 'Ada matching_key kosong'
assert not df_raw['matching_key'].duplicated().any(), 'Identitas desa terduplikasi. Periksa sumber!'
assert (df_raw['status_terverifikasi_JADESTA'].str.upper() == 'BELUM').all(), 'Status verifikasi berubah; periksa ulang metodologi.'

summary = df_raw.groupby(['label_3kelas','status_benchmark']).size().reset_index(name='jumlah')
summary['persentase'] = (summary['jumlah'] / len(df_raw) * 100).round(2)
display(summary)
summary.to_csv(OUT_DIR/'01_distribusi_label.csv', index=False, encoding='utf-8-sig')
print('Total identitas unik:', df_raw['matching_key'].nunique())
print('PENTING: Seluruh label adalah benchmark sementara, bukan verifikasi JADESTA.')

## Tahap 2 — Pembersihan data, *feature engineering*, dan TF-IDF input

Digunakan **12 fitur terstruktur** serta **1 kelompok fitur teks** yang akan diubah TF-IDF **di dalam pipeline model**, sehingga kosakata TF-IDF tidak mempelajari set uji atau fold validasi. `matching_key`, nama desa, kabupaten, nomor urut, label, status, dan kolom provenance **dikecualikan dari prediktor**. Fitur digital hanya menghitung kanal yang masuk akal untuk profil desa (bukan domain umum JADESTA/WhatsApp).

Jumlah fasilitas/produk adalah **indikator jumlah entri terdokumentasi**, bukan jumlah fasilitas yang telah diverifikasi di lapangan. Teks placeholder tidak dihitung sebagai entri aktual.

In [ ]:
EMPTY_VALUES = {'', '-', '--', '---', 'nan', 'none', 'null', 'n/a', 'na', 'tidak ada', 'belum ada', 'tdk ada'}
PSEUDO = {
    'Fasilitas': {'fasilitas', 'belum ada fasilitas untuk ditampilkan', 'belum tersedia'},
    'Atraksi': {'atraksi wisata', 'atraksi', 'belum ada atraksi untuk ditampilkan'},
    'homestay': {'kamar homestay', 'homestay', 'belum ada homestay untuk ditampilkan'},
    'pakert wisata': {'paket wisata', 'belum ada paket wisata untuk ditampilkan'},
    'sovenir': {'suvenir', 'souvenir', 'belum ada suvenir untuk ditampilkan'},
}

def clean_text(value):
    if pd.isna(value):
        return ''
    text = str(value).strip()
    if text.lower() in EMPTY_VALUES:
        return ''
    # Hapus ekspresi penghargaan/ranking yang berpotensi membocorkan label
    text = re.sub(r'\b\d+\s*besar\s*adwi\s*(?:20\d{2})?\b', ' ', text, flags=re.I)
    text = re.sub(r'\badwi\s*(?:20\d{2})?\b', ' ', text, flags=re.I)
    text = re.sub(r'\b(?:status|klasifikasi|kategori)\s+(?:desa\s+wisata\s+)?(?:mandiri|maju|rintisan|berkembang)\b', ' ', text, flags=re.I)
    return re.sub(r'\s+', ' ', text).strip()

def count_documented_items(value, column):
    text = clean_text(value)
    if not text or 'belum ada' in text.lower() or 'belum tersedia' in text.lower():
        return 0
    # Harga dalam format Rp 150,000 tidak boleh dihitung sebagai 2 entri
    text = re.sub(r'(?<=\d),(?=\d{3}(?:\D|$))', '', text)
    parts = re.split(r'[,;\n\r|]+', text)
    items = []
    for item in parts:
        item = item.strip(' .-_')
        if not item or item.lower() in PSEUDO[column]:
            continue
        for heading in PSEUDO[column]:
            if item.lower().startswith(heading + ' ') and len(item) > len(heading) + 2:
                item = item[len(heading):].strip()
        if item and item.lower() not in PSEUDO[column]:
            items.append(item)
    return len(items)

def valid_channel(value, channel):
    s = clean_text(value)
    if not s:
        return 0
    if channel == 'HP':
        return int(len(re.sub(r'\D', '', s)) >= 8)
    if channel == 'Email':
        return int(bool(re.search(r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b', s)))
    url = s if re.match(r'^https?://', s, flags=re.I) else 'https://' + s
    parsed = urlparse(url)
    host = parsed.netloc.lower().removeprefix('www.')
    slug = parsed.path.strip('/').lower()
    if channel == 'IG':
        return int(host in ('instagram.com','instagr.am') and slug not in ('','p','reel','stories','explore','jadestaofficial'))
    if channel == 'FB':
        return int(host in ('facebook.com','m.facebook.com','fb.com') and slug not in ('','jadestaofficial'))
    if channel == 'Twitter':
        return int(host in ('twitter.com','x.com') and slug not in ('','jadestaofficial','home','search'))
    if channel == 'Website':
        banned = ('facebook.com','instagram.com','twitter.com','x.com','wa.me','whatsapp.com',
                  'wa.link','linktr.ee','jadesta.kemenpar.go.id','jadesta.kemenparekraf.go.id')
        return int(bool(host and '.' in host and not any(host == d or host.endswith('.'+d) for d in banned)))
    return 0

NUM_COLS = ['description_length','num_facilities','num_attractions','num_homestay',
            'num_tour_packages','num_souvenirs','has_phone','has_instagram',
            'has_facebook','has_email','has_website','has_twitter']
TEXT_COL = 'combined_text_model'
TEXT_SOURCE_COLS = ['Deskripsi','Fasilitas','Atraksi','homestay','pakert wisata','sovenir']

df = df_raw.copy()
for col in required_columns:
    df[col] = df[col].fillna('').astype(str)
for col in TEXT_SOURCE_COLS:
    df['clean_' + col] = df[col].map(clean_text)

df['description_length'] = df['clean_Deskripsi'].map(lambda x: len(re.findall(r'\b\w+\b', x)))
for col, feature in [('Fasilitas','num_facilities'),('Atraksi','num_attractions'),
                     ('homestay','num_homestay'),('pakert wisata','num_tour_packages'),('sovenir','num_souvenirs')]:
    df[feature] = df[col].map(lambda x: count_documented_items(x, col))
for col, feature in [('HP','has_phone'),('IG','has_instagram'),('FB','has_facebook'),
                     ('Email','has_email'),('Website','has_website'),('Twitter','has_twitter')]:
    df[feature] = df[col].map(lambda x: valid_channel(x, col))

df[TEXT_COL] = df[['clean_' + col for col in TEXT_SOURCE_COLS]].agg(' '.join,axis=1).str.replace(r'\s+',' ',regex=True).str.strip()
for col in NUM_COLS:
    df[col] = pd.to_numeric(df[col], errors='raise').astype(float)
df['digital_presence_score'] = df[['has_instagram','has_facebook','has_email','has_website','has_twitter']].sum(axis=1)
df['product_readiness_score'] = df[['num_attractions','num_tour_packages','num_souvenirs']].sum(axis=1)
df['infrastructure_readiness_score'] = df[['num_facilities','num_homestay']].sum(axis=1)
assert df[NUM_COLS+[TEXT_COL]].notna().all().all()
assert (df[NUM_COLS].to_numpy() >= 0).all()

X_ALL = df[NUM_COLS+[TEXT_COL]].copy()
y_all = df['label_3kelas'].astype(int)
print('Matriks mentah prediktor:',X_ALL.shape,'| 12 numerik + 1 blok teks')
display(df.groupby('status_benchmark')[NUM_COLS].agg(['mean','median']).round(2).iloc[:,:12])
df[['matching_key','label_3kelas']+NUM_COLS+['digital_presence_score','product_readiness_score','infrastructure_readiness_score']].to_csv(
    OUT_DIR/'02_fitur_terstruktur_audit.csv',index=False,encoding='utf-8-sig')

## Tahap 3 — Stratified Holdout 80:20 (dikunci)

Pembagian menggunakan tiga label dengan seed 42. Setelah split, seluruh konfigurasi model, model terpilih, dan threshold diputuskan **tanpa melihat `y_test`**. Ukuran sampel Mandiri pada set uji sangat kecil (7), sehingga seluruh hasilnya harus dibaca dengan ketidakpastian tinggi.

In [ ]:
train_idx, test_idx = train_test_split(df.index.to_numpy(), test_size=0.20, stratify=y_all, random_state=SEED)
train_idx, test_idx = np.sort(train_idx), np.sort(test_idx)
assert len(set(train_idx) & set(test_idx)) == 0
train = df.loc[train_idx].copy()
test = df.loc[test_idx].copy()
X_train = train[NUM_COLS + [TEXT_COL]].copy()
X_test = test[NUM_COLS + [TEXT_COL]].copy()
y_A_train = (train['label_3kelas'] >= 1).astype(int)
y_A_test = (test['label_3kelas'] >= 1).astype(int)
train_B = train[train['label_3kelas'].isin([1,2])].copy()
test_B = test[test['label_3kelas'].isin([1,2])].copy()
X_B_train = train_B[NUM_COLS + [TEXT_COL]].copy()
X_B_test = test_B[NUM_COLS + [TEXT_COL]].copy()
y_B_train = (train_B['label_3kelas'] == 2).astype(int)
y_B_test = (test_B['label_3kelas'] == 2).astype(int)

split_table = pd.concat([
    train['label_3kelas'].value_counts().sort_index().rename('train_80'),
    test['label_3kelas'].value_counts().sort_index().rename('test_20')], axis=1).fillna(0).astype(int)
split_table['total'] = split_table.sum(axis=1)
split_table['kelas'] = split_table.index.map(LABEL_NAMES)
display(split_table)
split_table.to_csv(OUT_DIR/'03_split_train_test.csv',encoding='utf-8-sig')
train[['matching_key','Desa Wisata','Kabupaten','label_3kelas']].to_csv(OUT_DIR/'03_train_ids.csv',index=False,encoding='utf-8-sig')
test[['matching_key','Desa Wisata','Kabupaten','label_3kelas']].to_csv(OUT_DIR/'03_test_ids.csv',index=False,encoding='utf-8-sig')

## Tahap 4 — Pipeline TF-IDF + scaler + 3 model

- `StandardScaler` untuk 12 fitur terstruktur.
- `TfidfVectorizer(ngram_range=(1,2), max_features=3000, min_df=2, max_df=0.95)` untuk gabungan teks destinasi.
- `LogisticRegression(class_weight='balanced')`.
- `CalibratedClassifierCV(Pipeline(TF-IDF + LinearSVC(...)), cv=3)` sehingga *kalibrasi SVM juga tidak melihat kosakata TF-IDF dari fold validasinya*.
- `RandomForestClassifier(class_weight='balanced_subsample')`.

Tidak ada XGBoost. Ketiga model menerima **prediktor yang sama**. Pipeline yang dipilih tahap A dan tahap B boleh berbeda; evaluasi dilakukan terhadap **sistem DSS hierarkis yang benar-benar dihasilkan**.

In [ ]:
def make_features():
    return ColumnTransformer([
        ('numeric', StandardScaler(), NUM_COLS),
        ('tfidf', TfidfVectorizer(ngram_range=(1,2), min_df=2, max_df=0.95,
                                  max_features=3000, sublinear_tf=True), TEXT_COL)
    ], remainder='drop', sparse_threshold=0.30)

def get_estimator(name):
    if name == 'Logistic Regression':
        return Pipeline([('features',make_features()),
            ('model',LogisticRegression(max_iter=2500,class_weight='balanced',solver='liblinear',random_state=SEED))])
    if name == 'Calibrated Linear SVM':
        inner = Pipeline([('features',make_features()),
            ('model',LinearSVC(class_weight='balanced',dual='auto',max_iter=6000,random_state=SEED))])
        return CalibratedClassifierCV(estimator=inner, method='sigmoid', cv=3)
    if name == 'Random Forest':
        return Pipeline([('features',make_features()),
            ('model',RandomForestClassifier(n_estimators=RF_TREES, min_samples_leaf=2,
                                            class_weight='balanced_subsample',n_jobs=-1,random_state=SEED))])
    raise ValueError('Model tidak terdaftar: '+name)

MODELS = ['Logistic Regression','Calibrated Linear SVM','Random Forest']
cv = StratifiedKFold(n_splits=CV_FOLDS,shuffle=True,random_state=SEED)
SCORING = {'pr_auc':'average_precision','roc_auc':'roc_auc','balanced_accuracy':'balanced_accuracy',
           'f1':'f1','precision':'precision','recall':'recall','accuracy':'accuracy'}
print('Model:',MODELS,'| folds:',CV_FOLDS)

## Tahap 5 — Bandingkan model secara terpisah untuk Tahap A dan B

Prioritas pemilihan model: **mean PR-AUC** dari validasi silang (minority class), kemudian mean F1 dan balanced accuracy sebagai pemutus kesamaan. Metode yang memberikan hasil terbaik pada Tahap A tidak harus sama dengan Tahap B. Tidak ada data testing dalam tahap ini.

In [ ]:
def compare_stage(X, y, stage):
    results = []
    for name in MODELS:
        print('CV tahap',stage,'—',name,flush=True)
        scores = cross_validate(get_estimator(name),X,y,cv=cv,scoring=SCORING,
                                n_jobs=1,error_score='raise',return_train_score=False)
        row = {'tahap':stage,'model':name}
        for metric in SCORING:
            values = scores['test_'+metric]
            row[metric+'_mean'] = float(values.mean())
            row[metric+'_std'] = float(values.std(ddof=1))
        results.append(row)
    return pd.DataFrame(results).sort_values(['pr_auc_mean','f1_mean','balanced_accuracy_mean'],ascending=False).reset_index(drop=True)

cv_A = compare_stage(X_train,y_A_train,'A: advanced vs outside')
cv_B = compare_stage(X_B_train,y_B_train,'B: mandiri vs maju')
cv_results = pd.concat([cv_A,cv_B],ignore_index=True)
cv_results.to_csv(OUT_DIR/'04_perbandingan_model_cv.csv',index=False,encoding='utf-8-sig')
display(cv_results[['tahap','model','pr_auc_mean','pr_auc_std','f1_mean','f1_std','balanced_accuracy_mean','recall_mean']].round(4))
BEST_A = str(cv_A.iloc[0]['model'])
BEST_B = str(cv_B.iloc[0]['model'])
print('Model terpilih A:',BEST_A,'| Model terpilih B:',BEST_B)

## Tahap 6 — Optimasi ambang klasifikasi menggunakan prediksi OOF dari training saja

`cross_val_predict` menghasilkan probabilitas *out-of-fold*; threshold yang memaksimalkan F1 dipilih **tanpa mengevaluasi data testing**. Nilai F1 dari OOF yang digunakan untuk memilih threshold adalah **kinerja pemilihan**, bukan estimasi pengujian independen.

In [ ]:
def tune_threshold_oof(estimator, X, y, stage):
    oof = cross_val_predict(get_estimator(estimator),X,y,cv=cv,method='predict_proba',n_jobs=1)[:,1]
    records = []
    for threshold in np.linspace(0.05,0.95,91):
        yp = (oof >= threshold).astype(int)
        records.append({'tahap':stage,'threshold':float(threshold),
                        'f1':float(f1_score(y,yp,zero_division=0)),
                        'recall':float(recall_score(y,yp,zero_division=0)),
                        'balanced_accuracy':float(balanced_accuracy_score(y,yp))})
    grid = pd.DataFrame(records)
    # Ties: balanced_accuracy, then threshold nearest 0.5 (deterministic).
    grid['tie_break'] = -(grid['threshold'] - 0.5).abs()
    best = grid.sort_values(['f1','balanced_accuracy','tie_break'],ascending=False).iloc[0]
    grid.drop(columns=['tie_break']).to_csv(OUT_DIR/f'05_grid_threshold_{stage}.csv',index=False,encoding='utf-8-sig')
    return float(best['threshold']),oof,grid

THRESHOLD_A, oof_A, grid_A = tune_threshold_oof(BEST_A,X_train,y_A_train,'A')
THRESHOLD_B, oof_B, grid_B = tune_threshold_oof(BEST_B,X_B_train,y_B_train,'B')
print(f'Threshold OOF A = {THRESHOLD_A:.2f}; threshold OOF B = {THRESHOLD_B:.2f}')
model_A = get_estimator(BEST_A).fit(X_train,y_A_train)
model_B = get_estimator(BEST_B).fit(X_B_train,y_B_train)
print('Model A dan B selesai dilatih pada training set saja.')

## Tahap 7 — Evaluasi independen pada holdout (yang belum dipakai untuk seleksi)

DSS menghasilkan kelas `0 = Di luar benchmark`, `1 = Maju`, `2 = Mandiri` melalui threshold hierarkis. Skor `P(Mandiri)` dihitung dari **dua model**, sehingga dijelaskan oleh Permutation SHAP nanti **tanpa surrogate XGBoost**. Laporkan confusion matrix, macro F1, balanced accuracy, metrik kelas Mandiri, PR-AUC dan Brier stage A/B. Dengan hanya 7 Mandiri pada holdout, laporkan interval kepercayaan recall.

In [ ]:
def predict_hierarchy(model_a,model_b,X,threshold_a=THRESHOLD_A,threshold_b=THRESHOLD_B):
    pa = np.asarray(model_a.predict_proba(X))[:,1]
    pb = np.asarray(model_b.predict_proba(X))[:,1]
    p0,p1,p2 = 1-pa,pa*(1-pb),pa*pb
    pred = np.where(pa<threshold_a,0,np.where(pb<threshold_b,1,2))
    prob_3 = np.column_stack([p0,p1,p2])
    assert np.allclose(prob_3.sum(axis=1),1.0,atol=1e-9)
    return pred,prob_3,pa,pb

def binary_metrics(y_true,p_pos,thresh,prefix):
    y_pred = (p_pos>=thresh).astype(int)
    return {'tahap':prefix,'threshold':thresh,'n_test':int(len(y_true)),
            'positif_test':int(y_true.sum()), 'accuracy':accuracy_score(y_true,y_pred),
            'balanced_accuracy':balanced_accuracy_score(y_true,y_pred),
            'precision':precision_score(y_true,y_pred,zero_division=0),
            'recall':recall_score(y_true,y_pred,zero_division=0),
            'f1':f1_score(y_true,y_pred,zero_division=0),
            'roc_auc':roc_auc_score(y_true,p_pos),'pr_auc':average_precision_score(y_true,p_pos),
            'brier':brier_score_loss(y_true,p_pos)}

pred_test, p_test, pa_test, pb_test = predict_hierarchy(model_A,model_B,X_test)
pb_test_cond = model_B.predict_proba(X_B_test)[:,1]
pa_test_binary = model_A.predict_proba(X_test)[:,1]
metrics_AB = pd.DataFrame([
    binary_metrics(y_A_test,pa_test_binary,THRESHOLD_A,'A: Maju/Mandiri vs luar'),
    binary_metrics(y_B_test,pb_test_cond,THRESHOLD_B,'B: Mandiri vs Maju')
])
metrics_AB.to_csv(OUT_DIR/'06_evaluasi_holdout_tahap_A_B.csv',index=False,encoding='utf-8-sig')
display(metrics_AB.round(4))

y_true_test = test['label_3kelas'].astype(int).to_numpy()
labels3 = [0,1,2]
cm = confusion_matrix(y_true_test,pred_test,labels=labels3)
metrics_3 = {
    'accuracy':float(accuracy_score(y_true_test,pred_test)),
    'balanced_accuracy':float(balanced_accuracy_score(y_true_test,pred_test)),
    'macro_f1':float(f1_score(y_true_test,pred_test,average='macro',zero_division=0)),
    'weighted_f1':float(f1_score(y_true_test,pred_test,average='weighted',zero_division=0)),
    'mandiri_precision':float(precision_score(y_true_test,pred_test,labels=[2],average='macro',zero_division=0)),
    'mandiri_recall':float(recall_score(y_true_test,pred_test,labels=[2],average='macro',zero_division=0)),
    'mandiri_f1':float(f1_score(y_true_test,pred_test,labels=[2],average='macro',zero_division=0)),
    'n_test_mandiri':int((y_true_test==2).sum())
}
display(pd.DataFrame([metrics_3]).round(4))
print(classification_report(y_true_test,pred_test,labels=labels3,target_names=[LABEL_NAMES[k] for k in labels3],zero_division=0))
pd.DataFrame([metrics_3]).to_csv(OUT_DIR/'06_evaluasi_holdout_hierarkis.csv',index=False,encoding='utf-8-sig')
pd.DataFrame(cm,index=[f'aktual_{i}' for i in labels3],columns=[f'pred_{i}' for i in labels3]).to_csv(
    OUT_DIR/'06_confusion_matrix_counts.csv',encoding='utf-8-sig')

fig,ax=plt.subplots(figsize=(6.5,5.5))
ConfusionMatrixDisplay(cm,display_labels=['Luar','Maju','Mandiri']).plot(ax=ax,cmap='Blues',values_format='d',colorbar=False)
ax.set_title('Holdout: Confusion Matrix DSS Hierarkis')
fig.tight_layout();fig.savefig(OUT_DIR/'06_confusion_matrix.png',dpi=220,bbox_inches='tight');plt.show()

# Interval recall Mandiri binomial exact (Clopper-Pearson), bukan interval akurasi umum.
n_mandiri = int((y_true_test==2).sum())
tp_mandiri = int(((y_true_test==2)&(pred_test==2)).sum())
ci_lower = 0.0 if tp_mandiri==0 else beta.ppf(0.025,tp_mandiri,n_mandiri-tp_mandiri+1)
ci_upper = 1.0 if tp_mandiri==n_mandiri else beta.ppf(0.975,tp_mandiri+1,n_mandiri-tp_mandiri)
print(f'Recall Mandiri {tp_mandiri}/{n_mandiri}; 95% exact CI=[{ci_lower:.3f}, {ci_upper:.3f}]')

holdout_predictions = test[['matching_key','Desa Wisata','Kabupaten','label_3kelas']].copy()
holdout_predictions['prediksi_label'] = pred_test
holdout_predictions['p_benchmark_tingkat_lanjut'] = pa_test
holdout_predictions['p_mandiri_kondisional'] = pb_test
holdout_predictions['p_mandiri_gabungan'] = p_test[:,2]
holdout_predictions.to_csv(OUT_DIR/'06_prediksi_holdout.csv',index=False,encoding='utf-8-sig')

## Tahap 8 — Model final untuk DSS dan *readiness scoring* (seluruh 1.336 desa)

Setelah menyimpan evaluasi holdout, kedua model terpilih **dilatih ulang pada seluruh data yang relevan**. Skor dan klasifikasi di tahap ini bersifat **deployment/in-sample**, bukan estimasi generalisasi. Level readiness menggunakan batas **heuristik 0–20–40–60–80–100** untuk prioritas eksploratif; batas tersebut bukan kategori resmi JADESTA dan memerlukan uji sensitivitas/validasi pakar.

In [ ]:
model_A_full = get_estimator(BEST_A).fit(X_ALL,(y_all>=1).astype(int))
subset_B_full = df[df['label_3kelas'].isin([1,2])]
X_B_full = subset_B_full[NUM_COLS+[TEXT_COL]]
y_B_full = (subset_B_full['label_3kelas']==2).astype(int)
model_B_full = get_estimator(BEST_B).fit(X_B_full,y_B_full)

pred_all,p_all,pa_all,pb_all = predict_hierarchy(model_A_full,model_B_full,X_ALL)
result = df[['matching_key','Desa Wisata','Kabupaten','status_benchmark','label_3kelas','status_terverifikasi_JADESTA']+NUM_COLS].copy()
result['prediksi_kelas_model'] = pred_all
result['score_tingkat_lanjut_0_100'] = (pa_all*100).round(4)
result['score_menuju_mandiri_0_100'] = (p_all[:,2]*100).round(4)
result['p_mandiri_kondisional'] = pb_all.round(6)
result['tingkat_kesiapan_eksploratif'] = pd.cut(
    result['score_menuju_mandiri_0_100'],bins=[-0.001,20,40,60,80,100.00001],
    labels=['Low','Emerging Potential','Moderate Potential','High','Very High']).astype(str)
result['catatan_label'] = 'Benchmark sementara; belum verifikasi status resmi JADESTA'

result.to_csv(OUT_DIR/'07_DSS_SCORES_1336_DESA.csv',index=False,encoding='utf-8-sig')
print('Distribusi tingkat kesiapan eksploratif:')
display(result['tingkat_kesiapan_eksploratif'].value_counts().to_frame('desa'))
print('Top di luar benchmark berdasarkan score menuju Mandiri:')
display(result[result['label_3kelas']==0].sort_values('score_menuju_mandiri_0_100',ascending=False)[
    ['Desa Wisata','Kabupaten','score_tingkat_lanjut_0_100','score_menuju_mandiri_0_100','tingkat_kesiapan_eksploratif']].head(15))

## Tahap 9 — Benchmark-gap recommendation (Maju dan Mandiri terpisah)

Untuk desa **di luar benchmark**, kesenjangan dihitung terhadap median indikator **Maju**. Untuk desa **Maju**, terhadap median **Mandiri**. Status **Mandiri** menjadi kelompok referensi, bukan sasaran rekomendasi percepatan mandiri. Fokus rekomendasi pada atribut tindakan yang didukung data profil (fasilitas, atraksi, produk, kanal digital). Kesenjangan adalah **kesenjangan dokumentasi yang teramati** dan **bukan bukti bahwa fasilitas fisik tidak ada**.

In [ ]:
ACTIONABLE = {
    'num_facilities': 'Verifikasi dan lengkapi pencatatan fasilitas layanan dasar, akses, sanitasi, dan sarana wisata.',
    'num_attractions': 'Inventarisasi, dokumentasikan, dan tingkatkan kualitas atraksi yang bisa dikunjungi.',
    'num_homestay': 'Audit ketersediaan serta mutu akomodasi/homestay; lengkapi informasi bila sudah tersedia.',
    'num_tour_packages': 'Kembangkan dan dokumentasikan paket wisata dengan informasi layanan yang jelas.',
    'num_souvenirs': 'Inventarisasi produk UMKM/suvenir lokal dan tampilkan informasi produknya.',
    'has_instagram': 'Periksa keterkinian akun Instagram resmi destinasi dan lengkapi bila belum tersedia.',
    'has_facebook': 'Lengkapi informasi akun Facebook resmi apabila relevan dengan pengunjung sasaran.',
    'has_website': 'Periksa dan lengkapi situs resmi destinasi dengan informasi layanan yang terkini.'
}
benchmark_medians = df.groupby('label_3kelas')[list(ACTIONABLE)].median().T
benchmark_medians.columns = [LABEL_NAMES.get(c,c) for c in benchmark_medians.columns]
benchmark_medians.to_csv(OUT_DIR/'08_median_benchmark.csv',encoding='utf-8-sig')
display(benchmark_medians)

recs = []
for i,row in result.iterrows():
    status = int(row['label_3kelas'])
    if status == 2:
        continue
    target_class = 1 if status == 0 else 2
    target_name = LABEL_NAMES[target_class]
    benchmark = df.loc[df['label_3kelas']==target_class,list(ACTIONABLE)].median()
    gaps = []
    for feature,action in ACTIONABLE.items():
        target = float(benchmark[feature]); current = float(row[feature])
        if target <= 0:
            continue  # Tidak masuk akal merekomendasikan menaikkan fitur dengan median benchmark 0
        gap = max(0,target-current)
        if gap <= 0:
            continue
        gap_normalized = gap/max(target,1.0)
        gaps.append((gap_normalized,gap,feature,action,current,target))
    for priority,item in enumerate(sorted(gaps,reverse=True)[:3],start=1):
        normalized,gap,feature,action,current,target = item
        recs.append({'matching_key':row['matching_key'],'Desa Wisata':row['Desa Wisata'],
                     'Kabupaten':row['Kabupaten'],'status_asal':row['status_benchmark'],
                     'tujuan_benchmark':target_name,'score_menuju_mandiri_0_100':row['score_menuju_mandiri_0_100'],
                     'prioritas':priority,'fitur':feature,'nilai_teramati':current,
                     'median_benchmark':target,'gap':gap,'gap_ternormalisasi':normalized,
                     'rekomendasi_indikatif':action,
                     'peringatan':'Wajib validasi manual/lapangan; status benchmark belum terverifikasi'})
recs_df = pd.DataFrame(recs)
recs_df.to_csv(OUT_DIR/'08_rekomendasi_top3_per_desa.csv',index=False,encoding='utf-8-sig')
priority_outside = result[result['label_3kelas']==0].sort_values('score_menuju_mandiri_0_100',ascending=False)
priority_outside.to_csv(OUT_DIR/'08_prioritas_desa_di_luar_benchmark.csv',index=False,encoding='utf-8-sig')
print('Jumlah rekomendasi spesifik:',len(recs_df))
display(recs_df.head(12))

## Tahap 10 — Permutation SHAP **langsung** pada skor akhir DSS (tanpa XGBoost)

Target SHAP adalah fungsi nyata `P_A(advanced|X) × P_B(Mandiri|advanced,X)`, yaitu kolom `score_menuju_mandiri_0_100` dibagi 100. **Dua model yang dijelaskan adalah model final yang benar-benar dilatih pada tahap deployment**, bukan XGBoost pengganti.

**Unit penjelasan:** 12 atribut terstruktur ditambah **satu blok informasi teks TF-IDF**. TF-IDF pada pengukuran ini ditukar sebagai **satu dokumen penuh**, bukan per token; plot beeswarm hanya mencakup 12 atribut numerik agar nilai ID teks tidak dianggap skala kuantitatif. Pemilihan latar/observasi secara *stratified* dicatat dan kepentingan global dirata-ratakan menggunakan **bobot prevalensi 987/315/34**, sehingga sampel kelas Mandiri yang ditambah untuk diagnosis tidak mendominasi hasil global.

Jalankan sekali untuk demonstrasi; untuk publikasi sebaiknya uji stabilitas terhadap latar SHAP, tingkatkan sampel, serta laporkan biayanya. Penjelasan SHAP bukan bukti kausalitas.

In [ ]:
try:
    import shap
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable,'-m','pip','install','shap>=0.46,<1'])
    import shap
print('SHAP version:',shap.__version__)

FEATURE_COLS = NUM_COLS+[TEXT_COL]
# Registry teks, agar masker SHAP bisa bekerja pada satu matriks numerik:
# ID teks adalah label kategorikal bagi SELURUH dokumen, bukan angka berurutan untuk diinterpretasi.
text_bank = pd.Index(X_ALL[TEXT_COL].fillna('').astype(str).unique())
text_id = {s:i for i,s in enumerate(text_bank)}

def to_shap_numeric(Xdf):
    z = Xdf[FEATURE_COLS].copy()
    z[TEXT_COL] = z[TEXT_COL].map(text_id)
    if z.isna().any().any():
        raise ValueError('Ada teks di luar bank atau fitur numerik kosong.')
    return z.astype(float)

def final_score_from_encoded(z):
    z = np.atleast_2d(np.asarray(z,dtype=float))
    raw = pd.DataFrame(z,columns=FEATURE_COLS)
    ids = np.rint(raw[TEXT_COL].to_numpy()).astype(int)
    if (ids<0).any() or (ids>=len(text_bank)).any():
        raise ValueError('ID teks di luar daftar yang valid.')
    raw[TEXT_COL] = text_bank.take(ids).astype(str)
    a = model_A_full.predict_proba(raw[FEATURE_COLS])[:,1]
    b = model_B_full.predict_proba(raw[FEATURE_COLS])[:,1]
    return a*b

z_all = to_shap_numeric(X_ALL)
original_p = result['score_menuju_mandiri_0_100'].to_numpy()/100
wrapper_p = final_score_from_encoded(z_all.iloc[:8].to_numpy())
assert np.allclose(wrapper_p,original_p[:8],atol=1e-5,rtol=1e-5), 'Wrapper SHAP tidak sesuai DSS'
print('PASS: probabilitas SHAP = probabilitas sistem DSS asli (selisih hanya pembulatan tampilan).')

def strat_sample_indices(per_class,seed):
    parts=[]
    for class_id,n in enumerate(per_class):
        pool=df.index[df['label_3kelas']==class_id]
        parts.extend(pd.Index(pool).to_series().sample(n=min(n,len(pool)),random_state=seed+class_id).tolist())
    return np.array(parts,dtype=int)

idx_bg = strat_sample_indices(SHAP_BG_PER_CLASS,SEED+100)
idx_exp = strat_sample_indices(SHAP_EXPLAIN_PER_CLASS,SEED+200)
z_bg = z_all.loc[idx_bg]
z_exp = z_all.loc[idx_exp]
D = len(FEATURE_COLS)
MAX_EVALS = SHAP_N_PERMUTATIONS*(2*D+1)
explainer = shap.Explainer(final_score_from_encoded,
    shap.maskers.Independent(z_bg,max_samples=len(z_bg)),algorithm='permutation',
    feature_names=FEATURE_COLS,seed=SEED)
print(f'Background={len(z_bg)}; explained={len(z_exp)}; permutations={SHAP_N_PERMUTATIONS}; max_evals={MAX_EVALS}')
explanations = explainer(z_exp,max_evals=MAX_EVALS)
reconstructed = explanations.base_values + explanations.values.sum(axis=1)
actual = final_score_from_encoded(z_exp.to_numpy())
assert np.allclose(reconstructed,actual,atol=1e-5), 'Aditivitas SHAP gagal!'
print('PASS: base value + sum(SHAP) = skor Mandiri DSS asli.')

## Tahap 11 — SHAP global, beeswarm dan penjelasan desa individu

`mean_abs_shap_population_weighted` dihitung melalui rata-rata per kelas pada sampel SHAP, kemudian dibobot sesuai prevalensi kelas pada dataset (987/1336, 315/1336, 34/1336). Dengan sampel kecil, grafik merupakan **analisis awal** dan harus diperkuat untuk publikasi.

In [ ]:
strata_exp = df.loc[idx_exp,'label_3kelas'].to_numpy()
class_priors = df['label_3kelas'].value_counts(normalize=True).to_dict()
weighted_importance = np.zeros(D)
for cl in sorted(class_priors):
    class_values = np.abs(explanations.values[strata_exp==cl])
    if len(class_values)==0:
        raise RuntimeError('Ada kelas tidak terwakili dalam sampel SHAP.')
    weighted_importance += class_priors[cl]*class_values.mean(axis=0)

shap_global = pd.DataFrame({
    'fitur':NUM_COLS+['TF-IDF (seluruh blok teks)'],
    'mean_abs_shap_population_weighted':weighted_importance,
    'mean_abs_shap_sample':np.abs(explanations.values).mean(axis=0)
}).sort_values('mean_abs_shap_population_weighted',ascending=False)
shap_global.to_csv(OUT_DIR/'09_shap_global_importance.csv',index=False,encoding='utf-8-sig')
display(shap_global)

fig,ax=plt.subplots(figsize=(10,6))
ax.barh(shap_global['fitur'].iloc[::-1],shap_global['mean_abs_shap_population_weighted'].iloc[::-1])
ax.set_xlabel('Mean |SHAP| (kontribusi probabilitas; berbobot populasi)')
ax.set_title('Permutation SHAP — Model DSS Hierarkis (deployment)')
fig.tight_layout();fig.savefig(OUT_DIR/'09_shap_global.png',dpi=220,bbox_inches='tight');plt.show()

shap.plots.beeswarm(explanations[:,:len(NUM_COLS)],max_display=len(NUM_COLS),show=False)
plt.title('Permutation SHAP: fitur terstruktur (tanpa ID teks)')
plt.tight_layout();plt.savefig(OUT_DIR/'09_shap_beeswarm_numerik.png',dpi=220,bbox_inches='tight');plt.show()

# Penjelasan lokal: kandidat dengan skor terbesar dari luar benchmark
local_idx = int(priority_outside.index[0])
z_local = z_all.loc[[local_idx]]
local_xai = explainer(z_local,max_evals=MAX_EVALS)
p_local = float(final_score_from_encoded(z_local.to_numpy())[0])
assert np.isclose(float(local_xai.base_values[0]+local_xai.values[0].sum()),p_local,atol=1e-5)
local_values = pd.DataFrame({'fitur':NUM_COLS+['TF-IDF (seluruh blok teks)'],
                             'shap_contribution_probability':local_xai.values[0]})
local_values['nilai_absolut'] = local_values['shap_contribution_probability'].abs()
local_values = local_values.sort_values('nilai_absolut',ascending=False)
local_values.to_csv(OUT_DIR/'09_shap_lokal_contoh.csv',index=False,encoding='utf-8-sig')
print('Desa contoh:',result.loc[local_idx,'Desa Wisata'],'| score Mandiri:',round(p_local*100,4))
display(local_values)
fig,ax=plt.subplots(figsize=(10,6))
ax.barh(local_values['fitur'].iloc[::-1],local_values['shap_contribution_probability'].iloc[::-1])
ax.axvline(0,color='black',linewidth=1)
ax.set_xlabel('Kontribusi SHAP terhadap probabilitas Mandiri (bertanda)')
ax.set_title('Local Permutation SHAP — '+str(result.loc[local_idx,'Desa Wisata'])[:48])
fig.tight_layout();fig.savefig(OUT_DIR/'09_shap_lokal.png',dpi=220,bbox_inches='tight');plt.show()

## Tahap 12 — Ringkasan eksperimen dan ZIP output

Mencatat model A/B, threshold yang dipilih tanpa test, versi library, distribusi, metrik holdout, konfigurasi SHAP, serta peringatan label. File output otomatis dibundel menjadi ZIP untuk memudahkan pelaporan ke reviewer. Notebook `.ipynb` **tidak otomatis dimasukkan** dalam ZIP hasil Colab; simpan notebook via menu **File → Download → Download .ipynb**.

In [ ]:
experiment_manifest = {
    'data_total':int(len(df)),
    'class_counts':{str(k):int(v) for k,v in df['label_3kelas'].value_counts().sort_index().items()},
    'label_status':'BELUM terverifikasi JADESTA; label kelas 0 = di luar benchmark',
    'random_state':SEED, 'fast_mode':FAST_MODE,'cv_folds':CV_FOLDS,
    'split':'stratified 80:20; threshold/model selection from training folds only',
    'model_A':BEST_A,'model_B':BEST_B,'threshold_A':THRESHOLD_A,'threshold_B':THRESHOLD_B,
    'tfidf':{'ngram_range':[1,2],'max_features':3000,'min_df':2,'max_df':0.95},
    'structured_features':NUM_COLS,
    'score':'P(A: Maju/Mandiri|X) * P(B: Mandiri|A,X) * 100; descriptive/proxy only',
    'threshold_readiness_levels':'0-20-40-60-80-100 heuristic and unvalidated',
    'holdout_metrics':{k:float(v) for k,v in metrics_3.items()},
    'holdout_mandiri_recall_exact_95_ci':[float(ci_lower),float(ci_upper)],
    'shap':{'mode':'deployment','target':'two-stage P(Mandiri) score (same model)',
            'background_counts':list(SHAP_BG_PER_CLASS),'explained_counts':list(SHAP_EXPLAIN_PER_CLASS),
            'max_evals':int(MAX_EVALS),'feature_units':'12 structured + 1 grouped whole-text TF-IDF',
            'global_weighting':'observed class prevalence'},
    'versions':{'python':platform.python_version(),'scikit_learn':sklearn_version,'shap':shap.__version__,
                'pandas':pd.__version__,'numpy':np.__version__}
}
with open(OUT_DIR/'10_manifest_metode.json','w',encoding='utf-8') as h:
    json.dump(experiment_manifest,h,indent=2,ensure_ascii=False)

with open(OUT_DIR/'README_HASIL.txt','w',encoding='utf-8') as h:
    h.write('DSS DESA WISATA — CATATAN PENELITIAN\n'
            'Dataset 0/1/2 = di luar benchmark / Maju / Mandiri. BELUM diverifikasi JADESTA.\n'
            'Model hierarkis A: luar vs maju+mandiri; B: maju vs mandiri.\n'
            'Model dan threshold ditentukan pada training; holdout dipakai sekali untuk laporan.\n'
            'Hasil semua desa = in-sample deployment, BUKAN validasi independen.\n'
            'Skor Mandiri adalah keluaran model, bukan status resmi, probabilitas transisi, atau dampak kausal.\n'
            'SHAP dijalankan langsung ke fungsi skor akhir A*B tanpa XGBoost.\n'
            'Lihat manifest JSON untuk konfigurasi.\n')

zip_path = OUT_DIR.parent/'HASIL_DSS_1336_SHAP.zip'
with zipfile.ZipFile(zip_path,'w',compression=zipfile.ZIP_DEFLATED) as zf:
    for path in sorted(OUT_DIR.iterdir()):
        if path.is_file():
            zf.write(path,arcname=path.name)
print('ZIP hasil:',zip_path.resolve(),'| jumlah file:',len(list(OUT_DIR.iterdir())))
try:
    from google.colab import files as colab_files
    colab_files.download(str(zip_path))
except ImportError:
    pass

## Panduan pelaporan ilmiah — khusus revisi reviewer IJITCS

- **Pertahankan konsistensi tujuan:** Dengan label 0/1/2 saat ini, sistem melakukan **benchmark-membership assessment**, bukan klasifikasi resmi empat tingkat JADESTA. Jika klasifikasi resmi seluruh desa tersedia, gunakan sebagai label yang diverifikasi dan ulangi eksperimen.
- **Tabel metode:** cantumkan distribusi 987/315/34, 80:20 split, 5-fold CV, `class_weight`, konfigurasi TF-IDF, dan dua tahap model A/B. Perhatikan bahwa **34 Mandiri** hanya menghasilkan **7 Mandiri** pada holdout.
- **Tabel hasil:** sajikan hasil CV (mean ± SD), threshold berbasis train-only OOF, evaluasi independen A/B, macro F1/confusion matrix tiga kelas, dan confidence interval recall kelas Mandiri.
- **Skor dan level kesiapan:** skor `P_A×P_B` dihitung dari model full-fit, **tidak otomatis terkalibrasi**, interval kategori bersifat heuristik. Hindari menyatakan nilai 80 berarti peluang empiris 80% menjadi Mandiri.
- **SHAP:** jelaskan direct Permutation SHAP terhadap model final, 12 fitur struktural + satu grup teks TF-IDF. Jangan mengutip pentingnya kata tertentu dari satu kelompok TF-IDF tanpa analisis token tambahan.
- **Rekomendasi:** gunakan median grup Maju/Mandiri dengan verifikasi lapangan/pakar; entri profil kosong berarti belum terdokumentasi, bukan pasti tidak tersedia.
- **Penelitian lanjutan:** verifikasi kategori JADESTA, evaluasi sensitivitas batas readiness, kalibrasi probabilitas, validasi eksternal, dan validasi pakar/instansi untuk rekomendasi.